In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

PALETTE = ["#1D9E75", "#534AB7", "#BA7517", "#2E75B6", "#D85A30"]
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

df = pd.read_csv("data/training_results.csv")
print(f"Loaded {len(df)} episodes")
print(df.tail(3))

In [ ]:
def smooth(values, window=100):
    return pd.Series(values).rolling(window, min_periods=1).mean().values

ep = df['episode'].values

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ep, df['tcr']*100, alpha=0.15, color=PALETTE[0])
ax.plot(ep, smooth(df['tcr']*100), color=PALETTE[0], linewidth=2, label='Smoothed')
ax.axhline(38.5, color=PALETTE[4], linestyle='--', linewidth=1.5, label='Baseline: 38.5%')
ax.set_title("Task Completion Rate over Training", fontweight='bold')
ax.set_xlabel("Episode")
ax.set_ylabel("TCR (%)")
ax.legend()
plt.tight_layout()
plt.savefig("data/fig9_tcr_training.png", dpi=100, bbox_inches='tight')
plt.show()
plt.close()
print("Saved.")

In [ ]:
# Reward chart
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ep, df['reward'], alpha=0.15, color=PALETTE[1])
ax.plot(ep, smooth(df['reward']), color=PALETTE[1], linewidth=2, label='Smoothed')
ax.axhline(0, color='gray', linestyle='--', linewidth=1)
ax.set_title("Episode Reward over Training", fontweight='bold')
ax.set_xlabel("Episode")
ax.set_ylabel("Total reward")
ax.legend()
plt.tight_layout()
plt.savefig("data/fig9b_reward.png", dpi=100, bbox_inches='tight')
plt.show()
plt.close()
print("Saved.")

In [ ]:
# Epsilon and loss
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ep, df['epsilon'], color=PALETTE[2], linewidth=2)
axes[0].fill_between(ep, df['epsilon'], alpha=0.2, color=PALETTE[2])
axes[0].set_title("Epsilon Decay", fontweight='bold')
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Epsilon")

axes[1].plot(ep, df['loss'], alpha=0.2, color=PALETTE[3])
axes[1].plot(ep, smooth(df['loss']), color=PALETTE[3], linewidth=2, label='Smoothed')
axes[1].set_title("Training Loss (Huber)", fontweight='bold')
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.savefig("data/fig9c_epsilon_loss.png", dpi=100, bbox_inches='tight')
plt.show()
plt.close()
print("Saved.")

In [ ]:
import json, copy, random, numpy as np, torch, torch.nn as nn

# ── Config ────────────────────────────────────────────────────────────────────
WORKDAY_MINUTES = 600
MAX_TASKS       = 20
N_FEATURES      = 7
STATE_SIZE      = 140
HIDDEN_1        = 256
HIDDEN_2        = 256
HIDDEN_3        = 128
R_ON_TIME       = +10.0
R_MISSED        = -15.0
R_REORDER       = -2.0
R_DEP_RESOLVED  = +5.0
R_VAGUE         = +3.0
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH      = "data/dqn_model.pth"

# ── DQN Network ───────────────────────────────────────────────────────────────
class DQNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(STATE_SIZE, HIDDEN_1), nn.ReLU(),
            nn.Linear(HIDDEN_1, HIDDEN_2), nn.ReLU(),
            nn.Linear(HIDDEN_2, HIDDEN_3), nn.ReLU(),
            nn.Linear(HIDDEN_3, MAX_TASKS),
        )
    def forward(self, x): return self.net(x)

# ── Load trained model ────────────────────────────────────────────────────────
net = DQNetwork().to(DEVICE)
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
net.load_state_dict(checkpoint['online_net'])
net.eval()
print(f"Model loaded from {MODEL_PATH}")
print(f"Device: {DEVICE}")

In [ ]:
# ── Encode functions ──────────────────────────────────────────────────────────
def encode_task(task, current_time, completed_ids):
    time_left        = max(task['deadline_min'] - current_time, 0)
    deadline_urgency = float(np.clip(1.0-(time_left/WORKDAY_MINUTES), 0.0, 1.0))
    duration_norm    = float(np.clip(task['duration_min']/240.0, 0.0, 1.0))
    priority_norm    = (task['priority']-1)/4.0
    unmet            = [d for d in task.get('dependencies',[]) if d not in completed_ids]
    has_dep          = 1.0 if unmet else 0.0
    is_vague         = 1.0 if task.get('vague', False) else 0.0
    time_pressure    = float(np.clip(task['duration_min']/max(time_left,1), 0.0, 1.0))
    return np.array([deadline_urgency, duration_norm, priority_norm,
                     has_dep, is_vague, time_pressure, 1.0], dtype=np.float32)

def encode_state(queue, current_time, completed_ids):
    state = np.zeros((MAX_TASKS, N_FEATURES), dtype=np.float32)
    for i, task in enumerate(queue[:MAX_TASKS]):
        state[i] = encode_task(task, current_time, completed_ids)
    return state.flatten()

def greedy_action(state, queue):
    mask = np.zeros(MAX_TASKS, dtype=bool)
    mask[:len(queue)] = True
    with torch.no_grad():
        s = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
        q = net(s).squeeze(0)
        m = torch.BoolTensor(mask).to(DEVICE)
        q[~m] = float('-inf')
        return int(q.argmax().item())

# ── Load test scenarios ───────────────────────────────────────────────────────
with open("data/test_scenarios.json") as f:
    test_scenarios = json.load(f)

print(f"Evaluating on {len(test_scenarios)} test scenarios...")
results = []

for scenario in test_scenarios:
    queue         = copy.deepcopy(scenario['tasks'])
    disruptions   = copy.deepcopy(scenario['disruptions'])
    current_time  = 0
    completed     = []
    missed        = []
    steps         = 0

    while queue and current_time < WORKDAY_MINUTES and steps < 200:
        steps += 1

        # Apply disruptions
        remaining = []
        for d in disruptions:
            if d['type'] == 'urgent_insertion':
                if current_time >= d.get('arrival_min', 0):
                    queue.append(copy.deepcopy(d['task']))
                else:
                    remaining.append(d)
            elif d['type'] == 'deadline_shift':
                for t in queue:
                    if t['id'] == d['target_task_id']:
                        t['deadline_min'] = d['new_deadline_min']
            elif d['type'] == 'task_cancellation':
                queue = [t for t in queue if t['id'] != d.get('target_task_id')]
        disruptions = remaining

        if not queue:
            break

        completed_ids = {t['id'] for t in completed}
        state  = encode_state(queue, current_time, completed_ids)
        action = greedy_action(state, queue)
        action = min(action, len(queue)-1)
        task   = queue.pop(action)

        # Dependency check
        if task.get('dependencies'):
            unmet = [d for d in task['dependencies'] if d not in completed_ids]
            if unmet:
                queue.append(task)
                current_time += 10
                continue

        finish = current_time + task['duration_min']
        if finish > WORKDAY_MINUTES:
            missed.append(task)
            continue

        current_time = finish
        on_time = finish <= task['deadline_min']
        completed.append({**task, 'finish_time': finish, 'on_time': on_time})

    missed.extend(queue)

    total   = len(completed) + len(missed)
    on_time = [t for t in completed if t.get('on_time')]
    tcr     = len(on_time) / total if total > 0 else 0.0
    devs    = [abs(t['finish_time']-t['deadline_min']) for t in completed]
    dd      = float(np.mean(devs)) if devs else float(WORKDAY_MINUTES)

    results.append({
        'scenario_id': scenario['scenario_id'],
        'complexity':  scenario['complexity'],
        'tcr':         tcr,
        'dd':          dd,
        'completed':   len(completed),
        'missed':      len(missed),
    })

df_test = pd.DataFrame(results)
df_test.to_csv("data/dqn_test_results.csv", index=False)

print()
print("=" * 55)
print("  DQN FINAL TEST RESULTS (200 scenarios)")
print("=" * 55)
print(f"  TCR              : {df_test['tcr'].mean()*100:.2f}%")
print(f"  Mean DD          : {df_test['dd'].mean():.2f} min")
print()
print("  By complexity:")
for c in ['low', 'medium', 'high']:
    s = df_test[df_test['complexity']==c]
    print(f"    {c:<8} TCR={s['tcr'].mean()*100:.1f}%  DD={s['dd'].mean():.1f}min")
print()
print("  vs Baselines:")
print(f"    FIFO           : 38.49%")
print(f"    Priority Queue : 37.79%")
print(f"    DQN            : {df_test['tcr'].mean()*100:.2f}%")
imp = df_test['tcr'].mean()*100 - 38.49
print(f"    Improvement    : {imp:+.2f} pp vs FIFO")
print("=" * 55)